# SQL-анализ воронки продаж (Ozon)

В этом ноутбуке мы строим воронку продаж с помощью SQL-запросов напрямую к базе PostgreSQL. Это демонстрирует навыки работы с реляционными базами данных и агрегацией больших объемов данных.

In [1]:
import os
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")

DB_URL = f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST', 'localhost')}:{os.getenv('DB_PORT', '5432')}/{os.getenv('DB_NAME', 'ozon_ab_test')}"
engine = create_engine(DB_URL)

def q(sql: str) -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

## 1. Общая воронка (без учета последовательности)

Сначала посмотрим, сколько уникальных пользователей вообще совершали каждое действие.

In [2]:
general_funnel = q("""
    SELECT 'Просмотр' AS stage, COUNT(DISTINCT user_id) AS users
    FROM events WHERE action_type IN ('page_view', 'view_description')
    UNION ALL
    SELECT 'В корзину' AS stage, COUNT(DISTINCT user_id) AS users
    FROM events WHERE action_type = 'to_cart'
    UNION ALL
    SELECT 'Покупка' AS stage, COUNT(DISTINCT user_id) AS users
    FROM orders;
""")
general_funnel

,stage,users
0,Просмотр,911347
1,В корзину,475006
2,Покупка,834137


## 2. Строгая воронка (Sequential Funnel)

Теперь построим правильную воронку: пользователь учитывается на шаге "Покупка" только если он реально прошел шаги "Просмотр" и "Корзина".

In [3]:
strict_funnel = q("""
    WITH
        view_users AS (
            SELECT DISTINCT user_id FROM events WHERE action_type IN ('page_view', 'view_description')
        ),
        cart_users AS (
            SELECT DISTINCT user_id FROM events WHERE action_type = 'to_cart'
        ),
        purchase_users AS (
            SELECT DISTINCT user_id FROM orders
        )
    SELECT
        (SELECT COUNT(*) FROM view_users) AS views,
        (SELECT COUNT(*) FROM cart_users WHERE user_id IN (SELECT user_id FROM view_users)) AS carts,
        (SELECT COUNT(*) FROM purchase_users
         WHERE user_id IN (
             SELECT user_id FROM cart_users
             WHERE user_id IN (SELECT user_id FROM view_users)
         )) AS purchases;
""")

strict_funnel['conv_view_to_cart'] = (strict_funnel['carts'] / strict_funnel['views'] * 100).round(2)
strict_funnel['conv_cart_to_purchase'] = (strict_funnel['purchases'] / strict_funnel['carts'] * 100).round(2)
strict_funnel

,views,carts,purchases,conv_view_to_cart,conv_cart_to_purchase
0,911347,474566,452458,52.07,95.34


## 3. Базовая конверсия для A/B-теста

Этот запрос понадобится нам в следующем ноутбуке для дизайна эксперимента. Считаем **строгую конверсию из просмотра в покупку**.

In [4]:
base_conv_df = q("""
    WITH view_users AS (
        SELECT DISTINCT user_id FROM events
        WHERE action_type IN ('page_view', 'view_description')
    ),
    cart_users AS (
        SELECT DISTINCT user_id FROM events WHERE action_type = 'to_cart'
    ),
    purchase_users AS (
        SELECT DISTINCT user_id FROM orders
    )
    SELECT
        (SELECT COUNT(*) FROM purchase_users
         WHERE user_id IN (SELECT user_id FROM cart_users
                           WHERE user_id IN (SELECT user_id FROM view_users)))::float /
        (SELECT COUNT(*) FROM view_users) AS base_conversion;
""")
print(f"Строгая конверсия из просмотра в покупку: {base_conv_df['base_conversion'].iloc[0]:.2%}")

Строгая конверсия из просмотра в покупку: 49.65%


## 4. Активность по дням недели

Проверим, как распределяются заказы по дням недели, используя `EXTRACT(ISODOW FROM ...)`, где 1 — понедельник, 7 — воскресенье.

In [5]:
dow_df = q("""
    SELECT
        EXTRACT(ISODOW FROM created_timestamp) AS day_of_week_num,
        COUNT(*) AS orders_count
    FROM orders
    GROUP BY 1
    ORDER BY 1;
""")

dow_map = {1: 'Пн', 2: 'Вт', 3: 'Ср', 4: 'Чт', 5: 'Пт', 6: 'Сб', 7: 'Вс'}
dow_df['day_name'] = dow_df['day_of_week_num'].map(dow_map)
dow_df

,day_of_week_num,orders_count,day_name
0,1.0,2612564,Пн
1,2.0,2538945,Вт
2,3.0,2810295,Ср
3,4.0,2629464,Чт
4,5.0,2757798,Пт
5,6.0,2715076,Сб
6,7.0,2822980,Вс


## 5. Медианное время до покупки

Рассчитаем медианное время между первым просмотром и первой покупкой в часах, используя оконную функцию `PERCENTILE_CONT`. Чтобы результат был сопоставим с EDA-ноутбуком, ограничим анализ тридцатью днями и потребуем, чтобы заказ был позже просмотра.

In [6]:
time_to_convert_df = q("""
    WITH first_view AS (
        SELECT user_id, MIN(timestamp) AS view_time
        FROM events
        WHERE action_type = 'page_view'
        GROUP BY user_id
    ),
    first_order AS (
        SELECT user_id, MIN(created_timestamp) AS order_time
        FROM orders
        GROUP BY user_id
    )
    SELECT
        PERCENTILE_CONT(0.5) WITHIN GROUP (
            ORDER BY EXTRACT(EPOCH FROM (o.order_time - v.view_time)) / 86400
        ) AS median_days_to_convert
    FROM first_view v
    JOIN first_order o ON v.user_id = o.user_id
    WHERE o.order_time > v.view_time
      AND EXTRACT(EPOCH FROM (o.order_time - v.view_time)) / 86400 <= 30;
""")
days = time_to_convert_df['median_days_to_convert'].iloc[0]
print(f"Медианное время до покупки: {days:.2f} дней")

Медианное время до покупки: 4.47 дней


## 6. Топ-10 самых активных пользователей

Выявим пользователей с наибольшим числом событий — они могут быть ботами или скрейперами. Это частая проверка на качество данных при парсинге логов.

In [7]:
top_users_df = q("""
    SELECT
        user_id,
        COUNT(*) AS events_count
    FROM events
    GROUP BY user_id
    ORDER BY events_count DESC
    LIMIT 10;
""")
top_users_df

,user_id,events_count
0,4898080,3856
1,1288651,3003
2,2651540,2996
3,3130751,2993
4,1468531,2931
5,4532620,2929
6,3092880,2834
7,4235750,2763
8,4237040,2734
9,852721,2728


**Вывод:** В данных присутствуют пользователи с аномально высокой активностью — тысячи событий, что значительно превышает типичный уровень. В реальном проекте их стоило бы проверить на ботов и, возможно, исключить из воронки. Для текущего исследования оставляем их, так как это не влияет на агрегированные метрики (медианы и квантили устойчивы к выбросам).